# Step 1: Retrieving data

## Pull data from yfinance. 

### In **yf.download**, we accept the parameters: 
  1. tickers= --> accepts a single string or a list of strings
  2. start= --> string start date inclusive in the format YYYY-MM-DD
  3. end= --> string end date inclusive in the formaat YYYY-MM-DD
  4. interval= --> bar size, i.e. each row has what length of time worth of data. If interval="1h", each row has one hour of price data. We generally work with 1d
  5. auto_adjust=True(default) --> for eg, if a company does a 2-for-1 stock split, it means every existing share doubles into 2 new ones. Stock price would be halved. If we have auto_adjust=False, it would look like the stock had a 50% crash. Autoadjust accounts for these fluctuations

  - yfinance also imports pandas internally, so the returned object from .download is actually a pandas dataframe, and we can use pandas methods on it

## Why use Close data?
We COULD use the other data (except Volume because it is not price related), but Close is standard because:
1. Represents the final agreed price of the day after all trading activity
2. It establishes the true benchmark for a stock's value until the next day

In [3]:
import yfinance as yf
import pandas as pd
import numpy as np

from statsmodels.tsa.stattools import coint
from statsmodels.regression.linear_model import OLS
from statsmodels.tools import add_constant

In [4]:
tickers = ["KO", "PEP"]
START, END = "2015-01-01", "2024-01-01"
df = yf.download(tickers, start=START, end=END) #returns a pandas dataframe object 
prices = df["Close"]


[*********************100%***********************]  2 of 2 completed


# Step 2: Data Cleaning

## A) Filling up NaN values with .ffill() and .dropna()
### .ffill()
- uses data from the previous day to fill up a NaN value. 
- If a stock didn't trade today, the last known price is the best estimate of the value, unless that last known price is from too long ago
- we can pass the parameter limit=. This sets a limit on the number of days to look back to since ffill by default looks back as far as possible to get the previous data
### .dropna()
- removes any NaN values


- here, we use .ffill() AND THEN .dropna(), but why not .dropna() first before .ffill()?
  1. By filling before dropping NaN values, we prevent rows where KO has a valid value but PEP does not from being removed, meaning less data is removed
  2. Initially, I wondered wouldn't it be better to NaN values dropped first, before we ffill since we can use a price from multiple days ago if the previous day's price is also NaN. However, this is unideal as a 2-day gap is generally suspicious enough that it is safer to remove the price than fabricate it

In [5]:
# forward filling and dropping NaN values
prices = prices.ffill()
prices = prices.dropna()

## B) Minimum History Requirement
- here for KO and PEP, we are running a ***cointegration statistical test***. 

### What is a Cointegration Statistical Test
- It is essentially testing "is the relationship between KO and PEP real or could it be random?"
- We need sufficient data to ensure the p-value our statistical test produces has enough statistical power, if not the the p-value would be inconclusive and it would be unwise to act on it since any relationship shown could be from chance without sufficient data points to prove it is not. 
- Generally there are 2 types of Cointegration tests, the Engle-Granger Test and the Johansen Test
  - To put it simply, the Engle-Granger Test is used for 2 variables, while the Johansen Test is used for analysis of multiple time series at the same time, i.e. more than two variables

### Why check for 2 years worth of trading data specifically?
- To cover a full market cycle with a reasonable chance to capture both bullish and bearish market conditions. A relationship that only holds in one regime is not useful
- We will be using the Engle-Granger Test which generally needs a few hundered observations to produce reliable p-values
- The relationship needs time to express itself. Pairs trading relies on the spread mean-reverting (returning to their long-term historical means over time). If I only have say 3 months of data, I might not have seen enough full cycles of the spread diverging and reverting to know its a real pattern
- Note the 2 years minimum is just the floor below which statistics become unreliable, we generally aim for at least 5 years worth of data

### In the code, why use the assert function?
- The program will terminate immediately when our quantity of data is insufficient, allowing us to adjust

In [6]:
# Minimum History Requirement check (I am doing a check for 5 years here to make my data more robust)
MIN_YEARS = 5
assert len(prices) >= 252 * MIN_YEARS, f"Need at least {MIN_YEARS} years of data"

#will produce 
# AssertionError: Need at least __ years of data
# if not enough data

## C) Synchronisation Test
- Here, we are checking that for each row, the trading dates for KO and PEP are the same
- This is important as further down, we are running code similar to what we have below:
```python
spread = prices["PEP"] - hedge_ratio * prices["KO"]
```
  - Pandas aligns by index automatically before doing the arithmetic. So if KO has a date that PEP doesn't, pandas would return NaN for that row rather than using the wrong price.
  - However, this means that misaligned dates would silently introduce NaNs into the spread, which would lead to issues downstream
- Generally, for large stocks in the same exchange, it is expected for the stocks to have the same date indices. However, this issue can be more prevalent in stocks traded in different exchanges, or for stocks which are smaller

### .index in a time series dataframe
- Usually, indices in a pandas dataframe are integers from 0 onwards, but when we load time series data like what we have here, the indices become dates in the form of YYYY-MM-DD, so we can naturally confirm the dates are the same by ensuring the indices are the same

### prices[ticker].index
- We are returning a Series (single column) of data from the ticker of our choice. In this case, the column we are returning is the column with the dates(indices)

In [7]:
#Synchronisation check
#Note .equals is a pandas function, but yfinance imports it internally and the Dataframe it returns is a panddas object
assert prices["KO"].index.equals(prices["PEP"].index), "Tickers have misaligned trading dates"

## D) Outlier Check
- Occassionally price data can contain anomalous spikes caused by a multitude of reasons, from data feed errors, to a flash crash that was later cancelled etc. These can look like a 20% move in a single day when nothing real happened
- We hence look at daily returns and flag anything which is statistically implausible

### Why are we checking if he deviation of returns from the mean is more than 5 x σ?
- In statistics:
  - ~68% of values fall between 1σ of the mean
  - ~95% within 2σ
  - ~99.7% within 3σ
  - ~99.99...% within 5σ
- This means a daily return that is 5σ away from the mean is super rare under normal conditions. It is either a data error or something major happened that day to cause such a legitimate major price change

### What is mask?
- A boolean DataFrame object with the same shape as returns and price, but every value is True/False. 
- we can do ```returns - mean ``` as the pandas DataFrame object is able to process it as each value in the table having the mean from its respective ticker subtracted from it
- A row with a True value is flagged as an outlier as it means the returns from that day exceeded 5σ and it is worth investigating
- We can then find the sum of the number of anomalous return values with the sum method acting on mask

In [ ]:
# Outlier check
returns = prices.pct_change() #converting each data in the table to percentage change
mean = returns.mean() #returning a 1rx2c Series. Contains the means of KO and PEP from the percentage change data
std = returns.std() #returning a 1rx2c Series. Contains the standard deviations (σ) of KO and PEP from the percentage change data
mask = (returns - mean).abs() > 5 * std 
print(mask.sum())

Ticker
KO     12
PEP     8
dtype: int64


Interestingly, the program flagged 12 anomalous dates for KO and 8 for PEP, we shall investigate further:

In [9]:
ko_outliers = returns["KO"][mask["KO"]]
pep_outliers = returns["PEP"][mask["PEP"]]

print("KO outliers:")
print(ko_outliers)
print("\nPEP outliers:")
print(pep_outliers)

KO outliers:
Date
2019-02-14   -0.084355
2019-07-23    0.060719
2020-03-09   -0.061527
2020-03-12   -0.096725
2020-03-16   -0.066227
2020-03-19   -0.067336
2020-03-20   -0.084389
2020-03-26    0.064407
2020-04-06    0.064796
2020-06-11   -0.063348
2020-11-09    0.063094
2022-05-18   -0.069626
Name: KO, dtype: float64

PEP outliers:
Date
2020-03-12   -0.111060
2020-03-13    0.104994
2020-03-16   -0.112672
2020-03-17    0.129366
2020-03-20   -0.114283
2020-03-24    0.082335
2020-03-26    0.068978
2022-05-18   -0.061963
Name: PEP, dtype: float64


Analysing each row data we have, we can notice that most of the data falls between 2019 to 2020. This is the peak period when the Covid-19 scare was first emerging. During this period, there is bound to be pessimistic trader sentiments, but also positive ones for when traders falsely assumed the economy may have gotten back on track quickly. However, if we analyse deeper, there are some anomalous dates worth investigating further:

- 14 February 2019 (KO)
  - This is way before the first case of Covid first emerged in Wuhan in around November 2019. This hence cannot be due to the Covid-19 pandemic and it is worth investingating
  - Researching deeper, we can find out that on this day at 6:55am EST, Coca-Cola reported its earnings and provided a pessimistic financial forecast for 2019, likely causing por trader sentiments as well, accounting for the anomalous fall in returns observed

- 23 July 2019 (KO)
  - Similarly, this is before any Covid-19 case first emerged
  - On this day, Coca Cola reported a highly successful Q2 2019 earnings, besting Wall Street consensus estimates, likely boosting trader sentiments which accounts for the anomalous increase in returns observed

- 18 May 2022 (KO & PEP)
  - This is towards the tail end of the Covid-19 pandemic, but it is strange that this is the only date in 2022, roughly a year and a half after the latest anomalous date raised before it
  - On this day, it can be seen that the U.S. stock market suffered its worst daily percentage drop since June 2020, driven by surging inflation and a looming recession. Many stocks were affected, not just KO and PEP

- For the remaining dates, they are all concentrated in 2020, with majority of the dates even being in March. It is safe to assume it is due to the early Covid-19 pandemic scares which led to poor trader sentiments. Any anomalous rise in returns could also be due to traders wrongly assuming the stock market would recover quickly, as mentioned earlier

For all the anomalous dates, it is still worth keeping them in our data set. This is because every single flagged date has a legitimate real-world explanation connected to genuine extreme market events.

Cases when it is indeed necessary to remove the data include:
- A stock showing a 50% drop and recovery within the same day (clearly erroneous)
- A price of $0 or negative appearing briefly
- A date showing a 30% move with absolutely no news or market context

 

# E) Stationary Check

## Stationary vs Non-Stationary 
- Non-Stationary means a series has a **unit root** , meaning a value today is just yesterday's value plus some random noise, **with no pullback towards any mean**. The random fluctuation of the value is known as a **random walk**, where the series can drift anymore whith no tendency to revert. **Stock prices are meant to behave this way.**
- Stationary means a series has **no unit root, but rather a mean-rewverting pull**. This means the series is pull back towards the mean rather than drifting freely. **Stock returns are meant to behave this way**
- A stationary data series also has a constant mean, variance and autocorrelation structure over time.

## What is the ADF Test (Augmented Dickey-Fuller)?
- Tests whether a time series is stationary or non-stationary
- Null hypothesis is that the series has a unit root and is non-stationary, while the alternate hypothesis is that the series has no unit root and is stationary
- If p < 0.05, we have statistical evidence the series is stationary and we can reject the null hypothesis

## Aim of this check
- This check acts as a data sanity check to ensure the data we have actually behaves like a stock
- The fact is that stock price data is non-stationary, while their return data is stationary
- If our data does not behave as such, our data is arguably not stock data, indicating that there is something wrong

## Why is returns data stationary, but raw price data not?
- Raw prices drift because each day's price builds on the previous day's price, so the gains accumulate
- Returns don't accumulate that way and each day's return is relatively independent
- Over some years, there's no reason for the average daily return to systematically drift upward or downward. Stock price growth is always by a certain percentage and it just oscillates around some small positive mean (since stocks generally trend up slowly over time, the mean return is slightly above zero, but it's stable).

## What is the output of the adfuller method?
- A tuple with 6 elements:
  1) Test statistic --> The more negative it is, the stronger the evidence against the null hypothesis. On its own it doesn't mean much, it needs to be interpreted relative to the critical values (element 5)
  2) p-value 
  3) Number of lags used --> Chosen automatically, accounts for autocorrelation in the series
  4) Number of observations --> number of data points used in the test after accounting for the lags
  5) Critical values (dictionary) --> Thresholds for the test statistic at each significance level 1%, 5%, 10%. Our test statistic can be compared at each of these critical values and we can reject the null hypothesis at each significance level
  6) Information criterion

## Extension
- Time series data which is non-stationary by default, but becomes stationary after being differenced exactly one time is data which is **integrated of order 1**, denoted as *I(1)* 
- Differencing means subtrating the previous value from the current value
- A time series which is already stationary is I(0), meaning our returns data is I(0)


In [12]:
from statsmodels.tsa.stattools import adfuller

for ticker in ["KO", "PEP"]:
    result = adfuller(prices[ticker].pct_change().dropna())
    print(f"{ticker} ADF p-value: {result[1]:.4f}")

KO ADF p-value: 0.0000
PEP ADF p-value: 0.0000


# Big heading
## Smaller heading
### Even smaller

- item one
- item two
  - nested item (indent with 2 spaces)

1. first
2. second
3. third

**bold**
*italic*
`inline code`

$z = \frac{x - \mu}{\sigma}$

```python
x = 1 + 1
```